# 02. Core BattINFO Flow: Linked Records, Publication, and Submission

This notebook demonstrates the main BattINFO authoring workflow for a simple commercial cell.

You will create:

- one `CellType`
- one physical `Cell`
- one `Test`
- one linked `Dataset`
- one publication package
- one submission package

Everything writes under `.battinfo/notebooks/02-linked-records`.


In [ ]:
from pathlib import Path

from battinfo import Workspace, quantity


In [ ]:
workspace = Workspace(root=Path('.battinfo/notebooks/02-linked-records'), clean=True)

dataset_file = workspace.root / 'inputs' / 'a123-cycle-life.csv'
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_file.write_text(
    'cycle,capacity_ah,voltage_v\n0,2.50,3.30\n1,2.48,3.28\n2,2.46,3.27\n',
    encoding='utf-8',
)

dataset_file


## 1. Author the linked BattINFO objects

This is the core public API path for canonical record authoring.


In [ ]:
cell_type = workspace.cell_type(
    manufacturer='A123',
    model='ANR26650M1-B',
    year=2012,
    country_of_origin='United States',
    format='cylindrical',
    chemistry='Li-ion',
    size_code='R26650',
    iec_code='IFpR26650',
    positive_electrode_basis='LFP',
    negative_electrode_basis='Graphite',
    specs={
        'nominal_capacity': quantity(2.5, 'Ah'),
        'nominal_voltage': quantity(3.3, 'V'),
        'diameter': quantity(26.0, 'mm'),
        'height': quantity(65.0, 'mm'),
    },
    source_file='A123__ANR26650M1-B.pdf',
)

cell = workspace.cell(
    cell_type,
    serial_number='a123-demo-001',
    batch_id='A123-DEMO-01',
    source_type='lab',
)

test = workspace.test(
    cell,
    kind='cycle_life',
    protocol='1C charge / 1C discharge',
    instrument='Biologic VSP-300',
    status='completed',
)

dataset = workspace.dataset(
    cell,
    title='A123 ANR26650M1-B cycle-life dataset',
    description='Small sandbox dataset created by notebook 02.',
    test=test,
    path=dataset_file,
    license='CC-BY-4.0',
)

{
    'cell_type_model': cell_type.model,
    'cell_serial_number': cell.serial_number,
    'test_kind': test.test_kind,
    'dataset_name': dataset.name,
}


## 2. Save the canonical records


In [ ]:
save_result = workspace.save(validation_policy='strict')

{
    'cell_type_count': save_result['index']['cell_type_count'],
    'cell_instance_count': save_result['index']['cell_instance_count'],
    'test_count': save_result['index']['test_count'],
    'dataset_count': save_result['index']['dataset_count'],
}


## 3. Build the publication package

This writes the Schema.org JSON-LD publication graph and the RO-Crate and DataCite companion files.


In [ ]:
publication_result = workspace.build_publication_package(dataset)

{
    'publish_path': publication_result['publish_path'],
    'ro_crate_path': publication_result['ro_crate_path'],
    'datacite_metadata_path': publication_result['datacite_metadata_path'],
    'triple_count': publication_result['triple_count'],
}


## 4. Build the submission package

This is the handoff artifact for the registry workflow.


In [ ]:
submission_result = workspace.build_submission_package(
    dataset,
    registry='digibatt/demo-linked-records',
    publisher_id='demo-lab',
    version='0.1.0',
    title='A123 ANR26650M1-B cycle-life dataset',
    description='Simple linked-record walkthrough from notebook 02.',
)

{
    'submission_package_path': submission_result['submission_package_path'],
    'validation_report_path': submission_result['validation_report_path'],
    'resource_count': submission_result['resource_count'],
    'artifact_count': submission_result['artifact_count'],
}
